# 38 — Distributions and sampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/038-distributions-and-sampling.ipynb)

A sample mean is one draw from a distribution of possible means. This notebook simulates thousands of studies, plots those means, and checks the measured spread against the standard error formula `sigma / sqrt(n)`.

In [ ]:
import numpy as np  # arrays and random sampling
rng = np.random.default_rng(0)  # a reproducible generator
pop = np.array([2., 4., 6., 8., 10.])  # the five-value population
print("mean", pop.mean())  # 6.0
print("sd", pop.std())  # 2.828
draws = rng.choice(pop, size=(20000, 4))  # 20,000 samples of size 4
means = draws.mean(axis=1)  # one mean per sample
print("measured SE", means.std())  # about 1.41
print("formula SE", pop.std() / np.sqrt(4))  # 1.414

## Why the two numbers agree

Each sample mean is itself random. Its spread is not the spread of the data: the population standard deviation is 2.83, but a mean of four draws varies by only about 1.41. The divisor is `sqrt(n)`, so precision improves slowly.

### The shape

Even though the population has only five values, the distribution of the mean is close to a bell. That is the central limit theorem at work, and it is why the 1.96 multiplier for a 95% interval is usable in practice.

In [ ]:
import matplotlib.pyplot as plt  # plotting
small = rng.choice(pop, size=(20000, 4)).mean(axis=1)  # n = 4
large = rng.choice(pop, size=(20000, 64)).mean(axis=1)  # n = 64
plt.figure(figsize=(7, 3))  # wide figure
plt.hist(small, bins=30, density=True, alpha=0.6, label="n = 4")  # wide spread
plt.hist(large, bins=30, density=True, alpha=0.6, label="n = 64")  # narrow spread
plt.axvline(pop.mean(), color="black", linestyle="--")  # true mean
plt.xlabel("sample mean")  # horizontal label
plt.ylabel("density")  # vertical label
plt.legend()  # name both samples
plt.title("bigger samples give tighter means")  # takeaway
plt.show()  # draw it

## Reading the histogram

Both histograms are centred on 6.0, so neither estimator is biased. The `n = 64` histogram is about eight times taller and correspondingly narrower: its standard error is `2.83 / 8 = 0.35` against `1.41` for `n = 4`.

### From spread to an interval

An approximate 95% interval is `estimate +/- 1.96 * SE`. At `n = 64` the half-width is `1.96 * 0.35 = 0.69`, so the interval is roughly `[5.3, 6.7]`.

In [ ]:
def standard_error(values):  # sample sd divided by root n
    values = np.asarray(values, dtype=float)  # accept lists too
    if values.size < 2:  # need a spread
        raise ValueError("need at least two values")  # refuse one point
    return float(values.std(ddof=1) / np.sqrt(values.size))  # the formula
print(standard_error([2, 4, 6, 8, 10]))  # 1.414
print(standard_error([2, 4, 6, 8, 10] * 4))  # 0.707, twice the data